# Tennis Match Prediction — XGBoost
End-to-end pipeline: data cleaning → leak-safe features → training → evaluation → manual inference.

In [1]:
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict, deque
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

## 1. Load & Clean Data

In [2]:
df = pd.read_csv("datasets/matches_final.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# -1 → NaN for ranks; recompute Rank_Diff
df[["Rank_1", "Rank_2"]] = df[["Rank_1", "Rank_2"]].replace(-1, np.nan)
df["Rank_Diff"] = df["Rank_1"] - df["Rank_2"]

print(f"Rows: {len(df):,} | {df['Date'].min().date()} → {df['Date'].max().date()}")

Rows: 63,751 | 2000-01-03 → 2024-09-24


In [3]:
work_df = df[[
    "Date", "Player_1", "Player_2", "Surface", "Series", "Round", "Best of",
    "Winner", "Rank_1", "Rank_2", "Rank_Diff",
]].copy()

print("Shape:", work_df.shape)

Shape: (63751, 11)


## 2. Feature Engineering (leak-safe)

All features computed from **prior matches only** — no future leakage.

In [4]:
def add_historical_features(df, form_window=10, surface_window=30, elo_k=32):
    """Chronological pass: recent form, surface win rate, Elo."""
    form  = defaultdict(lambda: deque(maxlen=form_window))
    surf  = defaultdict(lambda: deque(maxlen=surface_window))
    elo   = defaultdict(lambda: 1500.0)

    rf1, rf2, sw1, sw2, el1, el2 = [], [], [], [], [], []

    for _, matches in df.groupby("Date", sort=True):
        # 1) Compute features from prior state
        for idx, r in matches.iterrows():
            p1, p2, s = r["Player_1"], r["Player_2"], r["Surface"]
            rf1.append((idx, np.mean(form[p1]) if form[p1] else 0.5))
            rf2.append((idx, np.mean(form[p2]) if form[p2] else 0.5))
            h1, h2 = surf[(p1, s)], surf[(p2, s)]
            sw1.append((idx, np.mean(h1) if h1 else 0.5))
            sw2.append((idx, np.mean(h2) if h2 else 0.5))
            el1.append((idx, elo[p1]))
            el2.append((idx, elo[p2]))

        # 2) Update state with results
        for _, r in matches.iterrows():
            p1, p2, s = r["Player_1"], r["Player_2"], r["Surface"]
            w1 = int(r["Winner"] == 1)
            form[p1].append(w1);      form[p2].append(1 - w1)
            surf[(p1, s)].append(w1); surf[(p2, s)].append(1 - w1)
            e1 = 1 / (1 + 10 ** ((elo[p2] - elo[p1]) / 400))
            elo[p1] += elo_k * (w1 - e1)
            elo[p2] += elo_k * ((1 - w1) - (1 - e1))

    df = df.copy()
    df["Recent_Form_1"]    = pd.Series(dict(rf1))
    df["Recent_Form_2"]    = pd.Series(dict(rf2))
    df["Recent_Form_Diff"] = df["Recent_Form_1"] - df["Recent_Form_2"]
    df["Surface_WR_1"]     = pd.Series(dict(sw1))
    df["Surface_WR_2"]     = pd.Series(dict(sw2))
    df["Surface_WR_Diff"]  = df["Surface_WR_1"] - df["Surface_WR_2"]
    df["Elo_1"]            = pd.Series(dict(el1))
    df["Elo_2"]            = pd.Series(dict(el2))
    df["Elo_Diff"]         = df["Elo_1"] - df["Elo_2"]
    return df

work_df = add_historical_features(work_df)
print("Features built:", work_df.shape)

Features built: (63751, 20)


In [5]:
feature_cols = [
    "Date", "Player_1", "Player_2", "Surface", "Series", "Round", "Best of",
    "Rank_1", "Rank_2", "Rank_Diff",
    "Recent_Form_1", "Recent_Form_2", "Recent_Form_Diff",
    "Surface_WR_1", "Surface_WR_2", "Surface_WR_Diff",
    "Elo_1", "Elo_2", "Elo_Diff", "Winner",
]
model_df = work_df[feature_cols].copy()
model_df.to_csv("datasets/features.csv", index=False)
print("Saved datasets/features.csv", model_df.shape)

Saved datasets/features.csv (63751, 20)


In [6]:
test = model_df[model_df["Date"] >= "2024-01-01"].dropna(subset=["Elo_Diff", "Rank_Diff"])
print(f"2024 matches: {len(test)}")
print(f"Higher-Elo  acc: {((test['Elo_Diff']  > 0) == (test['Winner'] == 1)).mean():.4f}")
print(f"Higher-Rank acc: {((test['Rank_Diff'] < 0) == (test['Winner'] == 1)).mean():.4f}")

2024 matches: 2189
Higher-Elo  acc: 0.6505
Higher-Rank acc: 0.6386


## 3. Train / Test Split & Model

In [7]:
num_cols = ["Rank_Diff", "Recent_Form_Diff", "Surface_WR_Diff", "Elo_Diff", "Best of"]
cat_cols = ["Surface", "Series", "Round"]

X = pd.get_dummies(model_df[num_cols + cat_cols].fillna(0), columns=cat_cols)
y = model_df["Winner"].astype(int)

train_mask = model_df["Date"] < "2024-01-01"
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (61562, 25) | Test: (2189, 25)


In [8]:
base_acc = accuracy_score(y_test, (model_df.loc[~train_mask, "Elo_Diff"] > 0).astype(int))
print(f"Baseline (higher Elo): {base_acc:.4f}")

model = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Trained.")

Baseline (higher Elo): 0.6505
Trained.


## 4. Evaluate

In [9]:
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba >= 0.5).astype(int)
acc   = accuracy_score(y_test, pred)

print(f"Accuracy: {acc:.4f}  (baseline {base_acc:.4f}, beats={acc > base_acc})")
print(f"Log-Loss: {log_loss(y_test, proba):.4f}")
print(f"Brier:    {brier_score_loss(y_test, proba):.4f}")

Accuracy: 0.6524  (baseline 0.6505, beats=True)
Log-Loss: 0.6115
Brier:    0.2127


## 5. Save Model

In [10]:
joblib.dump({"model": model, "columns": list(X.columns)}, "datasets/model.pkl")
print("Saved datasets/model.pkl")

Saved datasets/model.pkl


## 6. Manual Prediction for New Matches

Provide pre-match stats (Elo, rank, form, surface WR) for both players. No pickle state needed.

In [11]:
def predict_manual(pA, pB, surface, series, round_, best_of,
                   elo_A, elo_B, rank_A, rank_B, form_A, form_B, wr_A, wr_B):
    """Predict a match from raw stats you type in. No pickle state needed."""
    bundle = joblib.load("datasets/model.pkl")
    mdl, cols = bundle["model"], bundle["columns"]

    row = {
        "Rank_Diff":         rank_A - rank_B,
        "Recent_Form_Diff":  form_A - form_B,
        "Surface_WR_Diff":   wr_A   - wr_B,
        "Elo_Diff":          elo_A  - elo_B,
        "Best of":           best_of,
        "Surface":           surface,
        "Series":            series,
        "Round":             round_,
    }
    X1 = pd.get_dummies(pd.DataFrame([row]),
                        columns=["Surface", "Series", "Round"])
    X1 = X1.reindex(columns=cols, fill_value=0)
    p = float(mdl.predict_proba(X1)[0, 1])

    print(f"\n{pA} vs {pB}  |  {surface} | {series} | {round_} | Bo{best_of}")
    print(f"  {pA}: Elo={elo_A} Rank={rank_A} Form={form_A} WR={wr_A}")
    print(f"  {pB}: Elo={elo_B} Rank={rank_B} Form={form_B} WR={wr_B}")
    print(f"  → {pA}: {p*100:.1f}%   |   {pB}: {(1-p)*100:.1f}%")
    print(f"  → Predicted winner: {pA if p >= 0.5 else pB}")
    return {pA: round(p, 4), pB: round(1 - p, 4)}

### 2025 Australian Open Final — Sinner vs Zverev
**Actual result:** Sinner won 6-3, 7-6(4), 6-3.

In [19]:
predict_manual(
    "Sinner J.", "Zverev A.",
    surface="Hard", series="Grand Slam", round_="The Final", best_of=5,
    elo_A=2325, elo_B=2100,
    rank_A=1,   rank_B=2,
    form_A=0.90, form_B=0.85,
    wr_A=0.85,   wr_B=0.80,
)


Sinner J. vs Zverev A.  |  Hard | Grand Slam | The Final | Bo5
  Sinner J.: Elo=2325 Rank=1 Form=0.9 WR=0.85
  Zverev A.: Elo=2100 Rank=2 Form=0.85 WR=0.8
  → Sinner J.: 74.2%   |   Zverev A.: 25.8%
  → Predicted winner: Sinner J.


{'Sinner J.': 0.7419, 'Zverev A.': 0.2581}

### 2026 ATP Finals — Zverev vs Jodar
**Actual result:** Zverev won 6-4, 6-2.

In [20]:
predict_manual(
    "Zverev A.", "Jodar R.",
    surface="Hard", series="ATP Finals", round_="Round Robin", best_of=3,
    elo_A=2107, elo_B=2001,
    rank_A=1,   rank_B=11,
    form_A=0.79, form_B=0.69,
    wr_A=0.75,   wr_B=0.65,
)


Zverev A. vs Jodar R.  |  Hard | ATP Finals | Round Robin | Bo3
  Zverev A.: Elo=2107 Rank=1 Form=0.79 WR=0.75
  Jodar R.: Elo=2001 Rank=11 Form=0.69 WR=0.65
  → Zverev A.: 64.0%   |   Jodar R.: 36.0%
  → Predicted winner: Zverev A.


{'Zverev A.': 0.6397, 'Jodar R.': 0.3603}

### 2026 ATP Finals — Alcaraz vs Tiafoe
**Actual result:** Alcaraz won 6-2, 6-2.

In [21]:
predict_manual(
    "Alcaraz C.", "Tiafoe F.",
    surface="Hard", series="ATP Finals", round_="Round Robin", best_of=3,
    elo_A=2344, elo_B=1947,
    rank_A=3,   rank_B=8,
    form_A=0.90, form_B=0.65,
    wr_A=0.87,   wr_B=0.67,
)


Alcaraz C. vs Tiafoe F.  |  Hard | ATP Finals | Round Robin | Bo3
  Alcaraz C.: Elo=2344 Rank=3 Form=0.9 WR=0.87
  Tiafoe F.: Elo=1947 Rank=8 Form=0.65 WR=0.67
  → Alcaraz C.: 84.0%   |   Tiafoe F.: 16.0%
  → Predicted winner: Alcaraz C.


{'Alcaraz C.': 0.8396, 'Tiafoe F.': 0.1604}

### 2026 ATP Finals — Shelton vs Fils
**Actual result:** Shelton won 6-2, 3-6, 6-3.

In [22]:
predict_manual(
    "Shelton B.", "Fils A.",
    surface="Hard", series="ATP Finals", round_="Round Robin", best_of=3,
    elo_A=2040, elo_B=1985,
    rank_A=4,   rank_B=7,
    form_A=0.68, form_B=0.63,
    wr_A=0.67,   wr_B=0.63,
)


Shelton B. vs Fils A.  |  Hard | ATP Finals | Round Robin | Bo3
  Shelton B.: Elo=2040 Rank=4 Form=0.68 WR=0.67
  Fils A.: Elo=1985 Rank=7 Form=0.63 WR=0.63
  → Shelton B.: 55.0%   |   Fils A.: 45.0%
  → Predicted winner: Shelton B.


{'Shelton B.': 0.5497, 'Fils A.': 0.4503}

### 2026 ATP Finals — Medvedev vs Cobolli

In [23]:
predict_manual(
    "Medvedev D.", "Cobolli F.",
    surface="Hard", series="ATP Finals", round_="Round Robin", best_of=3,
    elo_A=1961, elo_B=1900,
    rank_A=5,   rank_B=6,
    form_A=0.73, form_B=0.65,
    wr_A=0.76,   wr_B=0.65,
)


Medvedev D. vs Cobolli F.  |  Hard | ATP Finals | Round Robin | Bo3
  Medvedev D.: Elo=1961 Rank=5 Form=0.73 WR=0.76
  Cobolli F.: Elo=1900 Rank=6 Form=0.65 WR=0.65
  → Medvedev D.: 58.5%   |   Cobolli F.: 41.5%
  → Predicted winner: Medvedev D.


{'Medvedev D.': 0.5853, 'Cobolli F.': 0.4147}

### 2026 ATP Finals — Fils vs Jodar (Tomorrow)

In [24]:
predict_manual(
    "Fils A.", "Jodar R.",
    surface="Hard", series="ATP Finals", round_="Round Robin", best_of=3,
    elo_A=1985, elo_B=2001,
    rank_A=7,   rank_B=11,
    form_A=0.63, form_B=0.69,
    wr_A=0.63,   wr_B=0.65,
)


Fils A. vs Jodar R.  |  Hard | ATP Finals | Round Robin | Bo3
  Fils A.: Elo=1985 Rank=7 Form=0.63 WR=0.63
  Jodar R.: Elo=2001 Rank=11 Form=0.69 WR=0.65
  → Fils A.: 47.0%   |   Jodar R.: 53.0%
  → Predicted winner: Jodar R.


{'Fils A.': 0.4705, 'Jodar R.': 0.5295}

### 2026 ATP Finals — Zverev vs Shelton (Tomorrow)

In [18]:
predict_manual(
    "Zverev A.", "Shelton B.",
    surface="Hard", series="ATP Finals", round_="Round Robin", best_of=3,
    elo_A=2107, elo_B=2040,
    rank_A=1,   rank_B=4,
    form_A=0.79, form_B=0.68,
    wr_A=0.75,   wr_B=0.67,
)


Zverev A. vs Shelton B.  |  Hard | ATP Finals | Round Robin | Bo3
  Zverev A.: Elo=2107 Rank=1 Form=0.79 WR=0.75
  Shelton B.: Elo=2040 Rank=4 Form=0.68 WR=0.67
  → Zverev A.: 59.4%   |   Shelton B.: 40.6%
  → Predicted winner: Zverev A.


{'Zverev A.': 0.5939, 'Shelton B.': 0.4061}